In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, avg,col,first,when,udf,regexp_replace,to_timestamp
from pyspark.sql.types import StringType, DoubleType, ArrayType
import re
ss = SparkSession.builder.config("spark.jars", "/home/jrodarte/postgresql-42.7.3.jar").getOrCreate()

In [22]:
dfMovimientos = ss.read.format("csv").options(header='true', inferSchema='true', delimiter=',').load("documentos/movimientos0725.csv")
#dfMovimientos.schema

In [23]:
# Datos de conexión
user = "jrodarte"
password = "roma1993_"

dfTipoMovimientos = ss.read.format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "(select * from prestadero.tipomovimientos)") \
    .option("user", user) \
    .option("password", password) \
    .option("driver", "org.postgresql.Driver") \
    .load()

#dfTipoMovimientos.show()

In [25]:
def limpiarImporte(importe):
    return importe.replace("$", "").replace(",", "")


limpiarImpoUDF = udf(limpiarImporte, StringType())

In [26]:
nuevosNombresMov = ['autorizacion', 'feoperacion', 'tipo', 'movimiento', 'importe', 'estatus', 'referencia']
nuevosNombresDetMov = ['autorizacion', 'detalle']
dfDetalleMovimiento = dfMovimientos.select('Autorización', 'Detalle')
dfMovimientos = dfMovimientos.selectExpr(
    "`Autorización`",
    "`Fecha operación`",
    "`Tipo`",
    "`Movimiento`",
    "`Importe`",
    "`Estatus`",
    "`Ref. 1`"
)

In [27]:
dfMovimientos = dfMovimientos.toDF(*nuevosNombresMov)
dfMovimientos = dfMovimientos.withColumn('feoperacion', to_timestamp(col("feoperacion"), "dd/MM/yyyy HH:mm:ss"))
dfMovimientos = dfMovimientos.withColumn('importeS', limpiarImpoUDF(col("importe")))
dfMovimientos = dfMovimientos.withColumn('importe', col("importeS").cast(DoubleType())).drop("importeS")
dfDetalleMovimiento = dfDetalleMovimiento.toDF(*nuevosNombresDetMov)
#dfMovimientos.show()
#dfMovimientos.schema

In [28]:
dfMovimientos = dfMovimientos.join(dfTipoMovimientos, dfMovimientos.movimiento == dfTipoMovimientos.tipomovimiento, "inner")
dfMovimientos = dfMovimientos.drop('tipomovimiento').drop('movimiento')
#dfMovimientos.schema

In [20]:
dfMovimientos.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "prestadero.movimientos") \
    .option("user", "jrodarte") \
    .option("password", "roma1993_") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [35]:
conteoMovimientos = dfMovimientos.count()
print(conteoMovimientos)

95


In [34]:
try:
    dfMovimientos.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "prestadero.movimientos") \
    .option("user", "jrodarte") \
    .option("password", "roma1993_") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()
except Exception as inst:
    print(type(inst))    # the exception type
    print(inst.args)     # arguments stored in .args
    print(inst)          # __str__ allows args to be printed directly,

<class 'py4j.protocol.Py4JJavaError'>
('An error occurred while calling o317.save.\n', JavaObject id=o318)
An error occurred while calling o317.save.
: org.postgresql.util.PSQLException: Connection to localhost:543 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:346)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:54)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:273)
	at org.postgresql.Driver.makeConnection(Driver.java:446)
	at org.postgresql.Driver.connect(Driver.java:298)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:50)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$creat